### Instruksi Pengerjaan

Buat notebook baru **`Tugas5_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, lalu:
1. Baca `transaksi_tugas5.csv` **dari HDFS** menjadi `df_transaksi`, tambahkan kolom `pendapatan` (`unit_terjual x harga_satuan`).
2. Buat `df_target` dari dictionary `data_target_cabang` di atas

Persiapan dan Inisialisasi Data

Jalankan kode berikut pada cell pertama untuk menginisialisasi SparkSession, membaca data transaksi dari HDFS, membuat kolom pendapatan, dan membuat dataframe referensi target.

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, row_number
from pyspark.sql.window import Window
import pandas as pd
import subprocess

# Mengambil konfigurasi default HDFS sistem secara otomatis untuk menghindari error URI
try:
    hdfs_uri = subprocess.check_output("hdfs getconf -confKey fs.defaultFS", shell=True).decode().strip()
    file_path = f"{hdfs_uri}/user/mahasiswa/tugas5/transaksi_tugas5.csv"
except:
    # Fallback jika subprocess gagal
    file_path = "hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv"

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas5_Praktikum") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 1. Membaca data dari HDFS
print(f"Mencoba membaca dari HDFS path: {file_path}")
df_transaksi = spark.read.csv(file_path, header=True, inferSchema=True)

# Menambahkan kolom pendapatan (unit_terjual dikali harga_satuan)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

# 2. Membuat DataFrame df_target
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

print("Data berhasil dimuat. Berikut 5 baris pertama df_transaksi:")
df_transaksi.show(5)

Mencoba membaca dari HDFS path: hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv
Data berhasil dimuat. Berikut 5 baris pertama df_transaksi:
+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows



**A. Join & Perbandingan Target** *(bobot 25%)*

Ringkas total `pendapatan` per `kota` dari `df_transaksi`, lalu **join** dengan `df_target`. Tambahkan kolom `pencapaian_persen`. Urutkan hasil dari pencapaian tertinggi.

In [8]:
# Meringkas total pendapatan per kota
ringkasan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Join dengan tabel target cabang
hasil_a = ringkasan_kota.join(df_target, on="kota", how="inner")

# Menambahkan kolom pencapaian_persen
hasil_a = hasil_a.withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan") * 100)
)

# Mengurutkan dari pencapaian tertinggi
hasil_a = hasil_a.orderBy(col("pencapaian_persen").desc())

print("--- Hasil Bagian A ---")
hasil_a.show()

--- Hasil Bagian A ---


[Stage 8:>                                                          (0 + 7) / 7]

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



**B. Window Function — Kategori Terlaris per Kota** *(bobot 25%)*

Menggunakan window function, tentukan **kategori dengan pendapatan tertinggi di setiap kota** (top-1 saja, gunakan `row_number()`).


In [9]:
# Agregasi total pendapatan untuk setiap kategori di masing-masing kota
pendapatan_kategori = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan_kategori")
)

# Mendefinisikan window: partisi per kota, urutkan berdasarkan pendapatan menurun
window_kategori = Window.partitionBy("kota").orderBy(col("total_pendapatan_kategori").desc())

# Memberikan nomor urut menggunakan row_number()
df_ranked = pendapatan_kategori.withColumn("urutan", row_number().over(window_kategori))

# Filter hanya urutan 1 (kategori terlaris per kota)
hasil_b = df_ranked.filter(col("urutan") == 1).drop("urutan")

print("--- Hasil Bagian B ---")
hasil_b.show()

--- Hasil Bagian B ---


[Stage 9:>                                                          (0 + 1) / 1]

+----------+--------------------+-------------------------+
|      kota|            kategori|total_pendapatan_kategori|
+----------+--------------------+-------------------------+
|  Magelang|Kesehatan & Kecan...|                  7275000|
| Purworejo|Kesehatan & Kecan...|                 10075000|
|  Semarang|        Rumah Tangga|                 11125000|
|      Solo|Kesehatan & Kecan...|                  8425000|
|Yogyakarta|             Fashion|                 13325000|
+----------+--------------------+-------------------------+



**C. Spark SQL** *(bobot 25%)*

Daftarkan `df_transaksi` dan `df_target` sebagai *temporary view*, lalu **tulis satu kueri SQL** (bukan DataFrame API) yang menampilkan: `kota`, `pic_cabang`, dan jumlah transaksi (`COUNT`) di kota tersebut, diurutkan dari jumlah transaksi terbanyak.


In [11]:
# Mendaftarkan temporary view
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

# Menulis kueri SQL murni
kueri_c = """
    SELECT t.kota, tar.pic_cabang, COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tar ON t.kota = tar.kota
    GROUP BY t.kota, tar.pic_cabang
    ORDER BY jumlah_transaksi DESC
"""

hasil_c = spark.sql(kueri_c)

print("--- Hasil Bagian C ---")
hasil_c.show()

--- Hasil Bagian C ---


[Stage 16:>                                                         (0 + 7) / 7]

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



**D. Kesimpulan** *(bobot 25%)*

Tulis pada markdown cell (**minimal 100 kata**): berdasarkan hasil bagian A dan B, **cabang mana yang berkinerja paling baik** dan **cabang mana yang paling perlu perhatian manajemen**? Sertakan angka-angka pendukung dari hasil analisis kalian, bukan opini tanpa dasar data.

Berdasarkan analisis data komprehensif pada Bagian A, dapat disimpulkan bahwa cabang **Purworejo** merupakan cabang dengan kinerja paling baik karena berhasil mencatatkan persentase pencapaian target bulanan tertinggi sebesar **152.16**%, di bawah tanggung jawab PIC **Fitri**. Keberhasilan yang signifikan ini didukung penuh oleh tingginya volume penjualan pada kategori **Kesehatan & Kecantikan** yang secara langsung menjadi penyumbang pendapatan terbesar di kota tersebut, sebagaimana yang dapat dilihat secara detail pada hasil Bagian B. Selain itu, Purworejo juga mencatatkan jumlah transaksi tertinggi dibandingkan cabang lainnya, yaitu sebanyak 116 transaksi.

Sebaliknya, cabang yang saat ini paling membutuhkan intervensi dan perhatian khusus dari pihak manajemen adalah cabang **Semarang**. Meskipun hanya dibebankan target penjualan sebesar Rp**55.000.000**, cabang ini hanya mampu merealisasikan pencapaian sebesar **69.41**% dari total target bulanannya. Manajemen disarankan untuk segera melakukan evaluasi menyeluruh bersama PIC **Sari** yang bertugas di cabang tersebut. Selain itu, diperlukan peninjauan ulang terhadap strategi promosi lokal, khususnya difokuskan pada kategori produk terlaris di wilayah tersebut (yaitu **Rumah Tangga**) agar dapat memaksimalkan potensi pasar yang ada dan mendongkrak performa penjualan ke depannya secara efektif.

**1. Eksplorasi DataFrame API: Pivot Table**

Pada Bagian B, kita hanya melihat 1 kategori terlaris per kota. Jika manajemen ingin melihat perbandingan seluruh kategori secara menyamping (kolom) untuk setiap kota, kita bisa menggunakan fungsi pivot(). Ini sangat berguna untuk membuat laporan matriks yang mudah dibaca.

In [12]:
# Membuat Pivot Table: Baris = Kota, Kolom = Kategori, Nilai = Total Pendapatan
df_pivot = df_transaksi.groupBy("kota") \
    .pivot("kategori") \
    .agg(spark_sum("pendapatan"))

print("--- Eksplorasi 1: Matriks Pendapatan per Kategori (Pivot) ---")
df_pivot.show()

--- Eksplorasi 1: Matriks Pendapatan per Kategori (Pivot) ---
+----------+----------+--------+----------------------+-----------------+------------+
|      kota|Elektronik| Fashion|Kesehatan & Kecantikan|Makanan & Minuman|Rumah Tangga|
+----------+----------+--------+----------------------+-----------------+------------+
|  Magelang|   7075000| 7200000|               7275000|          5225000|     4875000|
|  Semarang|   4500000| 4875000|               8475000|          9200000|    11125000|
|      Solo|   7350000| 4350000|               8425000|          7750000|     5600000|
| Purworejo|   9500000| 7575000|              10075000|          9600000|     8900000|
|Yogyakarta|  10500000|13325000|               7375000|          9200000|     6875000|
+----------+----------+--------+----------------------+-----------------+------------+



**2. Eksplorasi Spark SQL: Average Order Value (AOV)**

Jumlah transaksi yang banyak (Bagian C) tidak selalu berbanding lurus dengan pendapatan yang tinggi. Metrik penting lainnya di e-commerce adalah AOV, yaitu rata-rata nominal uang yang dihabiskan pembeli dalam satu kali transaksi di tiap cabang.

In [13]:
# Kueri untuk menghitung Average Order Value (AOV)
kueri_aov = """
    SELECT t.kota, 
           tar.pic_cabang, 
           COUNT(t.order_id) AS total_transaksi,
           SUM(t.pendapatan) AS total_pendapatan,
           ROUND(SUM(t.pendapatan) / COUNT(t.order_id), 2) AS rata_rata_nilai_transaksi
    FROM transaksi t
    JOIN target tar ON t.kota = tar.kota
    GROUP BY t.kota, tar.pic_cabang
    ORDER BY rata_rata_nilai_transaksi DESC
"""

hasil_aov = spark.sql(kueri_aov)

print("--- Eksplorasi 2: Rata-rata Nilai Transaksi (AOV) per Cabang ---")
hasil_aov.show()

--- Eksplorasi 2: Rata-rata Nilai Transaksi (AOV) per Cabang ---


[Stage 34:=================================>                        (4 + 3) / 7]

+----------+----------+---------------+----------------+-------------------------+
|      kota|pic_cabang|total_transaksi|total_pendapatan|rata_rata_nilai_transaksi|
+----------+----------+---------------+----------------+-------------------------+
|Yogyakarta|      Joko|            110|        47275000|                429772.73|
|  Semarang|      Sari|             93|        38175000|                410483.87|
| Purworejo|     Fitri|            116|        45650000|                393534.48|
|  Magelang|      Rani|             86|        31650000|                368023.26|
|      Solo|      Bayu|             95|        33475000|                352368.42|
+----------+----------+---------------+----------------+-------------------------+



In [14]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
